# Module 6: Store Agent Memory as a Graph in Neo4j

This module tests how an application stores, isolates, and traces agent memory. The booking agent stays unchanged.

The notebook stores one confirmed hotel preference in Neo4j. It recalls that preference for the same actor in a new session. A second actor receives zero preferences. The final query traces the preference to its source message, source session, and existing `Hotel` node. Internal reasoning stays out of the graph.

**Overview**

- **Short-term message:** Stores one user or assistant turn in a session.
- **Preference:** Stores a confirmed statement about what an actor wants.
- **Actor scope:** Starts a read at one `User` and returns that actor's memory.
- **Entity link:** Connects a source message to the canonical `Hotel` it mentions.
- **Provenance:** Connects a preference to its source message, session, and `Hotel`.
- **Embedding:** Uses Titan Text Embeddings V2 to create a vector for the saved preference.

**Access controls**

- **Scoped write:** Every library memory write includes a user identifier. The application must authenticate the actor and authorize access to each session ID.
- **Scoped read:** Version 0.5.0 semantic search covers the full memory store. This notebook uses Cypher that starts at the selected `User`.

**Prerequisites**

- **Hotel data:** The graph must contain exactly one `Hotel` named `AnyCompany Cairo Nile View`.
- **Neo4j access:** Configure the same Neo4j instance and database used by the earlier modules.
- **AWS access:** Configure credentials that can invoke Titan Text Embeddings V2 in `AWS_REGION`.
- **Configuration order:** `load_config` keeps existing environment values. It fills missing values from this folder's `.env`, the repository `.env`, and then `CONFIG.txt`.
- **Missing credentials:** Live cells print a skip message when Neo4j or AWS credentials are unavailable.

## Set up the notebook

The next cell checks the Python dependencies. Vocareum already includes them.

In [ ]:
# Vocareum already includes these dependencies. Run this cell as written.
# In another environment, uncomment the install command first.
# !pip install "neo4j-agent-memory[bedrock]==0.5.0" boto3 python-dotenv

print("Environment ready")

## 1. Create isolated IDs for this run

The next two cells load the workshop code and create the IDs used by every later step.

- **Run ID:** Adds a new eight-character value to this run's records.
- **Actor IDs:** Separate Alice's memory from Blake's memory.
- **Session IDs:** Separate Alice's first and second sessions from Blake's session.
- **`memory06-` prefix:** Gives cleanup one shared namespace for all Module 6 runs.

Each live step opens its own memory client and closes it when the step ends. The `finally` block also closes the client after an error.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("06-neo4j-memory")
print(f"Workshop root: {REPO_ROOT}")

In [ ]:
import uuid
from contextlib import asynccontextmanager

import boto3

from memory_helpers import (
    DEMO_ID_PREFIX,
    HERO_HOTEL_NAME,
    WORKSHOP_OWNER,
    build_memory_client,
    get_actor_preferences_for_hotel,
    link_message_to_hotel,
    link_preference_to_message_and_hotel,
    load_config,
    tag_demo_records,
)

RUN_ID = uuid.uuid4().hex[:8]
ACTOR_A = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-alice"
ACTOR_B = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-blake"
SESSION_A1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a1"
SESSION_A2 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a2"
SESSION_B1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-b1"

try:
    config = load_config()
except RuntimeError as exc:
    config = None
    print(f"Neo4j is not configured: {exc}")

MEMORY_READY = (
    config is not None and boto3.Session().get_credentials() is not None
)

@asynccontextmanager
async def open_memory():
    memory = build_memory_client(config)
    await memory.connect()
    try:
        yield memory
    finally:
        await memory.close()

if MEMORY_READY:
    print(f"Run {RUN_ID}: Neo4j at {config.uri}, Bedrock in {config.region}.")
else:
    print("Not configured. Every live cell below will skip.")

**Run the remaining cells once for each run ID.** To start a new run, execute the cell that defines `RUN_ID` again. Then rerun the later cells in order. Rerunning a message cell with the same run ID adds duplicate `Message` records.

## 2. Store a fixed hotel preference scenario

Store the same two-message conversation in every run. New run IDs keep the records separate.

The next cell performs four actions.

1. Check that the graph contains exactly one matching `Hotel`.
2. Store Alice's preference statement as a user message.
3. Store the assistant response in the same session.
4. Add `(:Message)-[:MENTIONS]->(:Hotel)` from the source message to the existing hotel.

The hotel has already been resolved to one canonical node. A production conversation can first use extraction to find a possible entity. When a tool returns a stable hotel identity, link that known hotel directly. This avoids an extra extraction step and prevents a duplicate domain node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        hotels = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (h:Hotel {name: $hotel_name})
            RETURN h.name AS name
            """,
            {"hotel_name": HERO_HOTEL_NAME},
        )
        if len(hotels) != 1:
            raise RuntimeError(
                f"The hotel graph must contain exactly one Hotel named "
                f"{HERO_HOTEL_NAME!r}; found {len(hotels)}."
            )

        preference_source = await memory.short_term.add_message(
            SESSION_A1,
            "user",
            f"I loved staying at {HERO_HOTEL_NAME}. A room on a high "
            "floor away from the elevator is a must for me.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_A1,
            "assistant",
            "I will remember that hotel and room preference.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )

    mentioned = link_message_to_hotel(
        config, str(preference_source.id), HERO_HOTEL_NAME
    )
    assert mentioned, "source message or unique hero Hotel was missing"
    print(f"Stored two fixture messages in {SESSION_A1}.")
    print("Linked the source message to the canonical Hotel.")

## 3. Promote the confirmed preference to durable memory

Store the confirmed statement as long-term memory. Entity linking identifies the hotel in the conversation. Memory policy separately decides which statement is safe and useful to keep. This fixed scenario treats Alice's statement as confirmed.

The next cell creates one `Preference` and adds its provenance.

- **`HAS_PREFERENCE`:** Connects Alice's `User` node to the preference.
- **`DERIVED_FROM`:** Connects the preference to its exact source message.
- **`MENTIONS`:** Connects the source message to the canonical hotel.
- **`ABOUT_HOTEL`:** Connects the preference to that same hotel.
- **Embedding:** Titan creates the preference vector during `add_preference`.

```text
(User)-[:HAS_PREFERENCE]->(Preference)-[:DERIVED_FROM]->(Message)-[:MENTIONS]->(Hotel)
                               |
                               +----------------[:ABOUT_HOTEL]----------->|
```

The code adds relationships to the existing `Hotel`. It leaves the node's labels and properties unchanged.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        preference = await memory.long_term.add_preference(
            category=f"hotels-{DEMO_ID_PREFIX}{RUN_ID}",
            preference=(
                f"Loves {HERO_HOTEL_NAME} and wants a room on a high "
                "floor away from the elevator."
            ),
            context=f"Workshop run {RUN_ID}, session {SESSION_A1}",
            user_identifier=ACTOR_A,
        )

    linked = link_preference_to_message_and_hotel(
        config,
        str(preference.id),
        str(preference_source.id),
        HERO_HOTEL_NAME,
    )
    assert linked, "preference, message, or unique hero Hotel was missing"
    print("Preference linked to its source message and the real Hotel.")

## 4. Read the preference in a new session and check a second actor

Confirm that Alice can recall her preference in a new session. Confirm that Blake receives zero results.

- **Alice:** Adds a message in `SESSION_A2`. The actor-scoped query returns her saved preference.
- **Blake:** Adds the same question in `SESSION_B1`. The same query returns zero preferences for him.
- **Production control:** Bind every actor ID and session ID to an authenticated caller before running the query.

The next cell creates both new-session messages and checks both query results.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        await memory.short_term.add_message(
            SESSION_A2,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_B1,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_B,
            extraction_mode="skip",
        )

    for_a = get_actor_preferences_for_hotel(
        config, ACTOR_A, HERO_HOTEL_NAME
    )
    for_b = get_actor_preferences_for_hotel(
        config, ACTOR_B, HERO_HOTEL_NAME
    )
    assert len(for_a) == 1, f"actor A expected one preference, got {len(for_a)}"
    assert not for_b, f"actor B unexpectedly saw {len(for_b)} preference(s)"
    print(f"Actor A in fresh session: {for_a[0]['preference']}")
    print("Actor B: no preference returned.")

## 5. Trace the preference to its source and hotel

Read the complete provenance path with one parameterized Cypher query. The query starts at Alice's `User` node and follows the preference to its source message, source session, and canonical `Hotel`. Its actor and hotel parameters keep the result within the requested scope.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        rows = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (u:User {identifier: $actor})
                  -[:HAS_PREFERENCE]->(p:Preference)
                  -[:DERIVED_FROM]->(m:Message)-[:MENTIONS]->(h:Hotel),
                  (m)<-[:HAS_MESSAGE]-(c:Conversation),
                  (p)-[:ABOUT_HOTEL]->(h {name: $hotel_name})
            RETURN u.identifier AS actor,
                   p.preference AS preference,
                   m.content AS source_message,
                   c.session_id AS source_session,
                   h.name AS hotel
            """,
            {"actor": ACTOR_A, "hotel_name": HERO_HOTEL_NAME},
        )
    assert len(rows) == 1, f"expected one provenance path, got {len(rows)}"
    for key, value in rows[0].items():
        print(f"{key:15s} {value}")

## 6. Mark this run for cleanup

Tag this run's memory records with the workshop owner value. Cleanup can then find data by either the `memory06-` namespace or the ownership marker. It removes matching memory records and workshop-owned relationships. It preserves every `Hotel` node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    marked = tag_demo_records(
        config,
        session_ids=[SESSION_A1, SESSION_A2, SESSION_B1],
        user_identifiers=[ACTOR_A, ACTOR_B],
    )
    print(f"Marked {marked} record(s) with {WORKSHOP_OWNER!r}.")

## 7. Compare managed memory and graph memory

Compare where each approach places extraction, storage, and operating responsibility.

| Dimension | AgentCore Memory | Neo4j graph memory in Module 6 |
|-----------|------------------|----------------------------------|
| Extraction | Runs as a managed asynchronous process | Follows application-controlled extraction and promotion rules |
| Availability | Appears after background processing | Appears when the write transaction commits |
| Inspection | Uses the Memory service API | Uses graph queries with source provenance |
| Correction | Uses the Memory service API | Adds a replacement and supersedes the old preference |
| Domain links | Keeps memory separate from domain data | Connects memory to the existing `Hotel` |
| Isolation | Manages actor namespaces | Uses scoped writes, actor-anchored reads, and application authorization |
| Operations | AWS operates the store | Your team operates Neo4j and the embedding contract |

- **Choose AgentCore Memory:** Use managed extraction and managed operations.
- **Choose Neo4j graph memory:** Control promotion, recall preferences immediately, trace their sources, and connect them to domain data.
- **Use both:** Let entity extraction identify the subject of a turn. Apply a separate memory policy to decide what becomes durable.

## 8. Optional: remove the workshop memory

The final cell leaves the completed exercise available for inspection by default.

- **Keep the data:** Leave `CLEAN_UP_DEMO_MEMORY` set to `False`.
- **Remove the data:** Set it to `True` and run the cell. Cleanup removes every Module 6 run in the `memory06-` namespace.
- **Preserve the graph:** Cleanup leaves all `Hotel` nodes in place.

In [ ]:
from cleanup_memory import run_cleanup

CLEAN_UP_DEMO_MEMORY = False

if config is None:
    print("Cleanup skipped: Neo4j is not configured.")
elif not CLEAN_UP_DEMO_MEMORY:
    print("Cleanup skipped. Set CLEAN_UP_DEMO_MEMORY = True when finished.")
else:
    cleanup_result = run_cleanup(config)
    assert cleanup_result == 0, f"cleanup returned {cleanup_result}"
